In [1]:

# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
from collections import deque
import math
import random
import json

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)


NumPy: 2.2.6
PyTorch: 2.3.1


In [2]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

from pathlib import Path
from collections import deque

import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# PROJECT DIRECTORIES
# ============================================================

TRAIN_RFENV_DIR = Path("./train_RFEnvs")
VAL_RFENV_DIR = Path("./val_RFEnvs")

ORACLE_DIR = Path("./Oracle")

TRAIN_ORACLE_DIR = Path("./Oracle")

VAL_ORACLE_DIR = Path("./val_oracle")

TRAIN_FINAL_DIR = Path("./train_final")

TRAIN_FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# RF REPRESENTATION
# ============================================================

FREQ_MIN_MHZ = 0

FREQ_MAX_MHZ = 18_000

FREQ_BIN_MHZ = 500

TIME_BIN_S = 0.050


N_FREQ_BINS = int(
    (
        FREQ_MAX_MHZ
        - FREQ_MIN_MHZ
    )
    / FREQ_BIN_MHZ
)


assert N_FREQ_BINS == 36


# ============================================================
# FROZEN RECURSIVE-MODEL CHECKPOINT
# ============================================================

RECURSIVE_MODEL_PATH = (
    TRAIN_FINAL_DIR
    / "recursive_belief_updater.pt"
)


# ============================================================
# BANDIT CHECKPOINT
#
# This is the temporal-Q checkpoint.
#
# Existing historical checkpoints are NOT overwritten.
# ============================================================

BANDIT_CHECKPOINT_PATH = (
    TRAIN_FINAL_DIR
    / "neural_oracle_bandit_temporal_q.pt"
)


# ============================================================
# TEMPORAL STATE
# ============================================================

SEQUENCE_LENGTH = 12

STATE_DIM = (
    7 * N_FREQ_BINS
    + 2
)

CANDIDATE_DIM = 7


# ============================================================
# NEURAL ARCHITECTURE
# ============================================================

GRU_HIDDEN_DIM = 128

ACTION_EMBED_DIM = 16

Q_HIDDEN_DIM = 128

N_BOOTSTRAP_HEADS = 5


# ============================================================
# OPTIMIZATION
#
# IMPORTANT:
# These names are required by BootstrappedNeuralBandit.
# ============================================================

LEARNING_RATE = 1e-3

WEIGHT_DECAY = 1e-4

GRAD_CLIP_NORM = 5.0


# ============================================================
# TRAINING
# ============================================================

SEED = 42

BATCH_SIZE = 128

REPLAY_CAPACITY = 20_000

TRAIN_UPDATES_PER_EPISODE = 4

MIN_REPLAY_TO_TRAIN = 256

BOOTSTRAP_PROB = 0.8


# ============================================================
# TEMPORAL Q-LEARNING
# ============================================================

GAMMA = 0.99

TARGET_TAU = 0.01


# ============================================================
# ORACLE SUPERVISION
# ============================================================

ORACLE_LOSS_WEIGHT_START = 1.0

ORACLE_LOSS_WEIGHT_MID = 0.5

ORACLE_LOSS_WEIGHT_END = 0.1

ORACLE_GUIDANCE_FULL_FRACTION = 0.60

ORACLE_GUIDANCE_END_FRACTION = 0.75

ORACLE_RANKING_MARGIN = 0.25


# ============================================================
# REWARD
#
# DO NOT CHANGE THESE FOR THE ARCHITECTURE EXPERIMENTS.
# ============================================================

REWARD_HIT_WEIGHT = 1.0

REWARD_NEW_BAND_WEIGHT = 0.75

REWARD_STALE_HIT_WEIGHT = 0.50

REWARD_REDUNDANCY_WEIGHT = 0.25

REWARD_INFORMATION_GAIN_WEIGHT = 0.0

REWARD_MISS_WEIGHT = 0.0


# ============================================================
# ACTION SELECTION
#
# The actual fixed Experiment-6 selector is implemented
# separately in:
#
#     29. CREATE FIXED POLICY + REWARD
#
# Keep the base policy configuration valid.
# ============================================================

THOMPSON_MODE = "mean_q"

SOFT_STALE = True

INITIAL_SWEEP = True

MAX_SCAN_GAP_STEPS = 40


# ============================================================
# RANDOM SEEDS
# ============================================================

random.seed(
    SEED
)

np.random.seed(
    SEED
)

torch.manual_seed(
    SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )


# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ============================================================
# SUMMARY
# ============================================================

print("=" * 70)
print("CONFIGURATION")
print("=" * 70)

print(
    f"Device                 : {DEVICE}"
)

print(
    f"Frequency bands        : {N_FREQ_BINS}"
)

print(
    f"State dimension        : {STATE_DIM}"
)

print(
    f"Candidate dimension    : {CANDIDATE_DIM}"
)

print(
    f"Sequence length        : {SEQUENCE_LENGTH}"
)

print(
    f"GRU hidden             : {GRU_HIDDEN_DIM}"
)

print(
    f"Q hidden               : {Q_HIDDEN_DIM}"
)

print(
    f"Bootstrap heads        : {N_BOOTSTRAP_HEADS}"
)

print(
    f"Learning rate          : {LEARNING_RATE}"
)

print(
    f"Weight decay           : {WEIGHT_DECAY}"
)

print(
    f"Replay capacity        : {REPLAY_CAPACITY}"
)

print(
    f"Gamma                  : {GAMMA}"
)

print(
    f"Target tau             : {TARGET_TAU}"
)

print(
    f"Oracle margin          : {ORACLE_RANKING_MARGIN}"
)

print(
    f"Initial sweep          : {INITIAL_SWEEP}"
)

print(
    f"Max scan gap           : {MAX_SCAN_GAP_STEPS}"
)

print(
    f"Selector               : {THOMPSON_MODE}"
)

print(
    f"Soft stale             : {SOFT_STALE}"
)

print(
    "\nReward weights:"
)

print(
    f"  Hit                  : "
    f"{REWARD_HIT_WEIGHT}"
)

print(
    f"  New band             : "
    f"{REWARD_NEW_BAND_WEIGHT}"
)

print(
    f"  Stale hit            : "
    f"{REWARD_STALE_HIT_WEIGHT}"
)

print(
    f"  Redundancy           : "
    f"{REWARD_REDUNDANCY_WEIGHT}"
)

print(
    f"  Information gain     : "
    f"{REWARD_INFORMATION_GAIN_WEIGHT}"
)

print(
    f"\nBandit checkpoint:"
)

print(
    f"  {BANDIT_CHECKPOINT_PATH}"
)

print(
    "\n✓ Configuration ready."
)

print(
    "✓ LEARNING_RATE defined."
)

print(
    "✓ WEIGHT_DECAY defined."
)

print("=" * 70)

CONFIGURATION
Device                 : cuda
Frequency bands        : 36
State dimension        : 254
Candidate dimension    : 7
Sequence length        : 12
GRU hidden             : 128
Q hidden               : 128
Bootstrap heads        : 5
Learning rate          : 0.001
Weight decay           : 0.0001
Replay capacity        : 20000
Gamma                  : 0.99
Target tau             : 0.01
Oracle margin          : 0.25
Initial sweep          : True
Max scan gap           : 40
Selector               : mean_q
Soft stale             : True

Reward weights:
  Hit                  : 1.0
  New band             : 0.75
  Stale hit            : 0.5
  Redundancy           : 0.25
  Information gain     : 0.0

Bandit checkpoint:
  train_final\neural_oracle_bandit_temporal_q.pt

✓ Configuration ready.
✓ LEARNING_RATE defined.
✓ WEIGHT_DECAY defined.


In [3]:

# ============================================================
# 3. LOAD RF ENVIRONMENTS
# ============================================================

def list_rf_environments(directory):
    directory = Path(directory)

    files = sorted(
        p for p in directory.glob("*.npy")
        if p.name != "environment_config.npy"
    )

    if not files:
        raise FileNotFoundError(
            f"No RF environments found in {directory.resolve()}"
        )

    return files


rf_files = list_rf_environments(
    TRAIN_RFENV_DIR
)

print(
    f"Found {len(rf_files)} training RF environments."
)

print(
    "First environment:",
    rf_files[0].name
)


Found 5 training RF environments.
First environment: config_0.npy


In [4]:

# ============================================================
# 4. TRANSMISSION EVENT EXTRACTION
# ============================================================

def extract_transmission_events(env):
    """
    A transmission event is a continuous run of 1s
    in one frequency band.
    """

    env = np.asarray(env)

    if env.ndim != 2:
        raise ValueError(
            f"Expected 2-D environment, got {env.shape}"
        )

    if env.shape[1] != N_FREQ_BINS:
        raise ValueError(
            f"Expected {N_FREQ_BINS} bands, got {env.shape[1]}"
        )

    events = []
    event_id = 0
    n_time_steps = len(env)

    for band in range(N_FREQ_BINS):

        in_event = False
        start = None

        for t in range(n_time_steps):

            active = bool(env[t, band] > 0)

            if active and not in_event:
                start = t
                in_event = True

            elif not active and in_event:

                end = t - 1

                events.append({
                    "id": event_id,
                    "band": band,
                    "start": start,
                    "end": end,
                    "duration": end - start + 1,
                })

                event_id += 1
                in_event = False

        if in_event:

            end = n_time_steps - 1

            events.append({
                "id": event_id,
                "band": band,
                "start": start,
                "end": end,
                "duration": end - start + 1,
            })

            event_id += 1

    events.sort(
        key=lambda e: (e["start"], e["band"])
    )

    for new_id, event in enumerate(events):
        event["id"] = new_id

    return events


def build_event_lookup(
    events,
    n_time_steps
):

    active_events = [
        [] for _ in range(n_time_steps)
    ]

    for event in events:

        for t in range(
            event["start"],
            event["end"] + 1
        ):

            active_events[t].append(
                event["id"]
            )

    return active_events


In [5]:

# ============================================================
# 6. LOAD THE FROZEN RECURSIVE BELIEF MODEL
# ============================================================

class BeliefUpdater(nn.Module):

    def __init__(
        self,
        n_bands=36,
        hidden_size=128
    ):

        super().__init__()

        self.n_bands = n_bands

        input_size = (
            n_bands
            + n_bands
            + 1
        )

        self.network = nn.Sequential(
            nn.Linear(
                input_size,
                hidden_size
            ),
            nn.ReLU(),

            nn.Linear(
                hidden_size,
                hidden_size
            ),
            nn.ReLU(),

            nn.Linear(
                hidden_size,
                n_bands
            )
        )

    def forward(
        self,
        probabilities,
        actions,
        observations
    ):

        action_one_hot = torch.zeros(
            actions.shape[0],
            self.n_bands,
            device=actions.device
        )

        action_one_hot.scatter_(
            1,
            actions.unsqueeze(1),
            1.0
        )

        observations = (
            observations
            .float()
            .unsqueeze(1)
        )

        x = torch.cat(
            [
                probabilities,
                action_one_hot,
                observations
            ],
            dim=1
        )

        return self.network(x)


checkpoint = torch.load(
    RECURSIVE_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

loaded_config = checkpoint[
    "model_config"
]

recursive_model = BeliefUpdater(
    n_bands=loaded_config[
        "n_freq_bins"
    ],
    hidden_size=loaded_config[
        "hidden_size"
    ]
).to(DEVICE)

recursive_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

recursive_model.eval()

for parameter in recursive_model.parameters():
    parameter.requires_grad = False

P_baseline = checkpoint[
    "P_baseline"
].astype(np.float32)

assert len(P_baseline) == N_FREQ_BINS

print("✓ Frozen recursive belief model loaded.")
print("Checkpoint:", RECURSIVE_MODEL_PATH)
print("Baseline shape:", P_baseline.shape)


✓ Frozen recursive belief model loaded.
Checkpoint: train_final\recursive_belief_updater.pt
Baseline shape: (36,)


In [6]:

# ============================================================
# 7. RECURSIVE BELIEF PROVIDER
# ============================================================

class RecursiveBeliefProvider:

    def __init__(
        self,
        model,
        baseline,
        device
    ):

        self.model = model
        self.baseline = np.asarray(
            baseline,
            dtype=np.float32
        )
        self.device = device

        self.reset()

    def reset(self):

        self.belief = torch.tensor(
            self.baseline,
            dtype=torch.float32,
            device=self.device
        ).unsqueeze(0)

    def __call__(
        self,
        t,
        action,
        observation
    ):

        if action is None:

            return (
                self.belief
                .squeeze(0)
                .detach()
                .cpu()
                .numpy()
                .copy()
            )

        action_tensor = torch.tensor(
            [action],
            dtype=torch.long,
            device=self.device
        )

        observation_tensor = torch.tensor(
            [observation],
            dtype=torch.float32,
            device=self.device
        )

        with torch.no_grad():

            logits = self.model(
                self.belief,
                action_tensor,
                observation_tensor
            )

            self.belief = torch.sigmoid(
                logits
            )

        return (
            self.belief
            .squeeze(0)
            .detach()
            .cpu()
            .numpy()
            .copy()
        )


belief_provider = RecursiveBeliefProvider(
    model=recursive_model,
    baseline=P_baseline,
    device=DEVICE
)

print("✓ Recursive belief provider ready.")


✓ Recursive belief provider ready.


In [7]:

# ============================================================
# 9. SCHEDULER MEMORY
# ============================================================

class SchedulerMemory:

    def __init__(
        self,
        n_bands=N_FREQ_BINS,
        hit_memory=10
    ):

        self.n_bands = n_bands
        self.hit_memory = hit_memory

        self.reset()

    def reset(self):

        self.scan_counts = np.zeros(
            self.n_bands,
            dtype=np.float32
        )

        self.hit_counts = np.zeros(
            self.n_bands,
            dtype=np.float32
        )

        self.age_since_scan = np.full(
            self.n_bands,
            np.inf,
            dtype=np.float32
        )

        self.time_since_hit = np.full(
            self.n_bands,
            np.inf,
            dtype=np.float32
        )

        self.last_observation = np.zeros(
            self.n_bands,
            dtype=np.float32
        )

        self.recent_observations = [
            deque(maxlen=self.hit_memory)
            for _ in range(self.n_bands)
        ]

        self.previous_action = -1
        self.step = 0

    def _normalized_age(self, values):

        values = np.asarray(
            values,
            dtype=np.float32
        )

        finite = np.isfinite(values)

        clipped = np.where(
            finite,
            np.minimum(
                values,
                float(MAX_SCAN_GAP_STEPS)
            ),
            float(MAX_SCAN_GAP_STEPS)
        )

        return (
            clipped
            / float(MAX_SCAN_GAP_STEPS)
        ).astype(np.float32)

    def hit_rates(self):

        rates = np.zeros(
            self.n_bands,
            dtype=np.float32
        )

        for band in range(self.n_bands):

            history = (
                self.recent_observations[
                    band
                ]
            )

            if history:

                rates[band] = (
                    float(sum(history))
                    / float(len(history))
                )

        return rates

    def state_vector(self, belief):

        belief = np.asarray(
            belief,
            dtype=np.float32
        )

        if belief.shape != (
            self.n_bands,
        ):

            raise ValueError(
                f"Expected belief shape "
                f"({self.n_bands},), "
                f"got {belief.shape}"
            )

        age = self._normalized_age(
            self.age_since_scan
        )

        time_since_hit = (
            self._normalized_age(
                self.time_since_hit
            )
        )

        hit_rates = self.hit_rates()

        scan_count = (
            np.log1p(
                self.scan_counts
            )
            / np.log1p(50.0)
        )

        scan_count = np.clip(
            scan_count,
            0.0,
            1.0
        )

        previous_action = np.zeros(
            self.n_bands,
            dtype=np.float32
        )

        if (
            0
            <= self.previous_action
            < self.n_bands
        ):

            previous_action[
                self.previous_action
            ] = 1.0

        phase = (
            2.0
            * np.pi
            * self.step
            / 200.0
        )

        time_features = np.asarray(
            [
                np.sin(phase),
                np.cos(phase),
            ],
            dtype=np.float32
        )

        state = np.concatenate(
            [
                belief,
                age,
                hit_rates,
                time_since_hit,
                self.last_observation,
                scan_count,
                previous_action,
                time_features,
            ]
        ).astype(np.float32)

        if len(state) != STATE_DIM:

            raise RuntimeError(
                f"State dimension mismatch: "
                f"{len(state)} != {STATE_DIM}"
            )

        return state

    def candidate_features(
        self,
        belief
    ):

        belief = np.asarray(
            belief,
            dtype=np.float32
        )

        age = self._normalized_age(
            self.age_since_scan
        )

        time_since_hit = (
            self._normalized_age(
                self.time_since_hit
            )
        )

        hit_rates = self.hit_rates()

        scan_count = (
            np.log1p(
                self.scan_counts
            )
            / np.log1p(50.0)
        )

        scan_count = np.clip(
            scan_count,
            0.0,
            1.0
        )

        band_position = (
            np.arange(
                self.n_bands,
                dtype=np.float32
            )
            / max(
                self.n_bands - 1,
                1
            )
        )

        features = np.stack(
            [
                belief,
                age,
                hit_rates,
                time_since_hit,
                self.last_observation,
                scan_count,
                band_position,
            ],
            axis=1
        ).astype(np.float32)

        assert features.shape == (
            self.n_bands,
            CANDIDATE_DIM
        )

        return features

    def reward_context(
        self,
        action
    ):

        age = self.age_since_scan[
            action
        ]

        stale_age = (
            float(MAX_SCAN_GAP_STEPS)
            if not np.isfinite(age)
            else float(age)
        )

        is_new_band_hit = (
            self.scan_counts[action] > 0
            and self.hit_counts[action] > 0
        )

        recent_scan_fraction = (
            max(
                0.0,
                1.0
                - stale_age
                / float(MAX_SCAN_GAP_STEPS)
            )
        )

        return {
            "age": stale_age,

            "time_since_hit": (
                float(MAX_SCAN_GAP_STEPS)
                if not np.isfinite(
                    self.time_since_hit[action]
                )
                else float(
                    self.time_since_hit[action]
                )
            ),

            "is_new_band_hit":
                bool(is_new_band_hit),

            "redundancy":
                float(recent_scan_fraction),
        }

    def update(
        self,
        action,
        observation
    ):

        hit = int(
            observation > 0
        )

        self.age_since_scan += 1

        finite_hit_age = np.isfinite(
            self.time_since_hit
        )

        self.time_since_hit[
            finite_hit_age
        ] += 1

        self.age_since_scan[
            action
        ] = 0

        self.scan_counts[
            action
        ] += 1

        self.hit_counts[
            action
        ] += hit

        self.last_observation[
            action
        ] = hit

        self.recent_observations[
            action
        ].append(hit)

        if hit:
            self.time_since_hit[
                action
            ] = 0

        self.previous_action = int(
            action
        )

        self.step += 1


In [8]:

# ============================================================
# 10. TEMPORAL STATE BUILDER
# ============================================================

class TemporalStateBuilder:

    def __init__(
        self,
        n_bands=N_FREQ_BINS,
        sequence_length=SEQUENCE_LENGTH
    ):

        self.n_bands = n_bands
        self.sequence_length = sequence_length

        self.memory = SchedulerMemory(
            n_bands=n_bands
        )

        self.history = deque(
            maxlen=sequence_length
        )

    def reset(self):

        self.memory.reset()
        self.history.clear()

    def observe_state(
        self,
        belief
    ):

        state = self.memory.state_vector(
            belief
        )

        candidate = (
            self.memory.candidate_features(
                belief
            )
        )

        self.history.append(
            state.copy()
        )

        sequence = list(
            self.history
        )

        if len(sequence) < (
            self.sequence_length
        ):

            first = sequence[0]

            padding = [
                first.copy()
                for _ in range(
                    self.sequence_length
                    - len(sequence)
                )
            ]

            sequence = (
                padding
                + sequence
            )

        sequence = np.stack(
            sequence,
            axis=0
        ).astype(np.float32)

        assert sequence.shape == (
            self.sequence_length,
            STATE_DIM
        )

        return sequence, candidate

    def update(
        self,
        action,
        observation
    ):

        self.memory.update(
            action,
            observation
        )

    def reward_context(
        self,
        action
    ):

        return self.memory.reward_context(
            action
        )


In [9]:

# ============================================================
# 12. SINGLE NEURAL BANDIT HEAD
# ============================================================

class NeuralBanditHead(nn.Module):

    def __init__(
        self,
        state_dim=STATE_DIM,
        n_bands=N_FREQ_BINS,
        candidate_dim=CANDIDATE_DIM,
        gru_hidden_dim=GRU_HIDDEN_DIM,
        action_embed_dim=ACTION_EMBED_DIM,
        q_hidden_dim=Q_HIDDEN_DIM,
    ):

        super().__init__()

        self.n_bands = n_bands

        self.gru = nn.GRU(
            input_size=state_dim,
            hidden_size=gru_hidden_dim,
            batch_first=True,
        )

        self.action_embedding = nn.Embedding(
            n_bands,
            action_embed_dim
        )

        self.q_network = nn.Sequential(

            nn.Linear(
                gru_hidden_dim
                + candidate_dim
                + action_embed_dim,
                q_hidden_dim
            ),

            nn.LayerNorm(
                q_hidden_dim
            ),

            nn.ReLU(),

            nn.Linear(
                q_hidden_dim,
                q_hidden_dim // 2
            ),

            nn.ReLU(),

            nn.Linear(
                q_hidden_dim // 2,
                1
            )
        )

    def encode(
        self,
        state_sequence
    ):

        _, hidden = self.gru(
            state_sequence
        )

        return hidden[-1]

    def forward(
        self,
        state_sequence,
        candidate_features
    ):

        latent = self.encode(
            state_sequence
        )

        batch_size = (
            state_sequence.shape[0]
        )

        latent = (
            latent
            .unsqueeze(1)
            .expand(
                -1,
                self.n_bands,
                -1
            )
        )

        band_ids = torch.arange(
            self.n_bands,
            device=state_sequence.device
        )

        band_embeddings = (
            self.action_embedding(
                band_ids
            )
        )

        band_embeddings = (
            band_embeddings
            .unsqueeze(0)
            .expand(
                batch_size,
                -1,
                -1
            )
        )

        x = torch.cat(
            [
                latent,
                candidate_features,
                band_embeddings,
            ],
            dim=-1
        )

        q_values = self.q_network(
            x
        ).squeeze(-1)

        return q_values


In [10]:

# ============================================================
# 21. ONLINE SCHEDULING SIMULATOR
# ============================================================

def run_policy_on_environment(
    env,
    policy,
    belief_provider,
    reward_function=None,
    training=False,
):
    """
    Run one complete RF environment.

    During the loop, the policy sees only:
        current belief
        scheduler memory
        selected-band observation

    The full RF environment is used only to:
        1. generate the selected observation
        2. calculate offline event metrics
    """

    env = np.asarray(
        env,
        dtype=np.float32
    )

    if env.ndim != 2:
        raise ValueError(
            f"Expected 2-D environment, got {env.shape}"
        )

    if env.shape[1] != N_FREQ_BINS:
        raise ValueError(
            f"Expected {N_FREQ_BINS} bands, got {env.shape[1]}"
        )

    n_time_steps = len(env)

    events = extract_transmission_events(
        env
    )

    active_events = build_event_lookup(
        events,
        n_time_steps
    )

    event_by_id = {
        event["id"]: event
        for event in events
    }

    policy.reset_episode(
        collect=training
    )

    belief_provider.reset()

    if reward_function is not None:
        reward_function.reset()

    intercepted_event_ids = set()
    interception_times = {}

    actions = []
    observations = []
    beliefs = []
    rewards = []

    belief = belief_provider(
        t=0,
        action=None,
        observation=None
    )

    for t in range(n_time_steps):

        # ----------------------------------------------------
        # 1. Policy chooses one band.
        # ----------------------------------------------------

        action = int(
            policy.select_action(
                belief
            )
        )

        if not (
            0 <= action < N_FREQ_BINS
        ):

            raise ValueError(
                f"Invalid action {action}"
            )

        belief_before = np.asarray(
            belief,
            dtype=np.float32
        ).copy()

        # ----------------------------------------------------
        # 2. Only selected band is observed.
        # ----------------------------------------------------

        observation = float(
            env[t, action]
        )

        # ----------------------------------------------------
        # 3. Frozen recursive belief update.
        # ----------------------------------------------------

        belief_after = belief_provider(
            t=t + 1,
            action=action,
            observation=observation
        )

        # ----------------------------------------------------
        # 4. Observable reward.
        # ----------------------------------------------------

        if reward_function is not None:

            reward = float(
                reward_function(
                    action=action,
                    observation=observation,
                    belief_before=belief_before,
                    belief_after=belief_after,
                    scheduler_context=(
                        policy.reward_context(
                            action
                        )
                    ),
                )
            )

        else:

            reward = 0.0

        # ----------------------------------------------------
        # 5. Policy records transition and updates memory.
        # ----------------------------------------------------

        policy.update(
            action,
            observation,
            reward
        )

        actions.append(action)
        observations.append(observation)
        beliefs.append(belief_before)
        rewards.append(reward)

        # ----------------------------------------------------
        # OFFLINE event evaluation
        # ----------------------------------------------------

        for event_id in active_events[t]:

            if event_id in intercepted_event_ids:
                continue

            event = event_by_id[
                event_id
            ]

            if event["band"] == action:

                intercepted_event_ids.add(
                    event_id
                )

                interception_times[
                    event_id
                ] = t

        belief = belief_after

    intercepted_events = []

    for event_id in intercepted_event_ids:

        event = event_by_id[
            event_id
        ]

        interception_time = (
            interception_times[event_id]
        )

        delay_steps = (
            interception_time
            - event["start"]
        )

        intercepted_events.append({
            **event,

            "interception_time":
                interception_time,

            "delay_steps":
                delay_steps,

            "delay_s":
                delay_steps * TIME_BIN_S,
        })

    missed_events = [
        event
        for event in events
        if event["id"]
        not in intercepted_event_ids
    ]

    n_events = len(events)
    n_intercepted = len(
        intercepted_events
    )

    interception_rate = (
        n_intercepted / n_events
        if n_events > 0
        else np.nan
    )

    if intercepted_events:

        delays = np.asarray([
            event["delay_s"]
            for event in intercepted_events
        ])

        mean_delay = float(
            np.mean(delays)
        )

        total_delay = float(
            np.sum(delays)
        )

    else:

        mean_delay = np.nan
        total_delay = 0.0

    unique_bands = len({
        event["band"]
        for event in intercepted_events
    })

    return {
        "events":
            events,

        "intercepted_events":
            intercepted_events,

        "missed_events":
            missed_events,

        "actions":
            np.asarray(actions),

        "observations":
            np.asarray(observations),

        "beliefs":
            np.asarray(beliefs),

        "rewards":
            np.asarray(rewards),

        "intercepted_event_count":
            n_intercepted,

        "total_event_count":
            n_events,

        "event_interception_rate":
            interception_rate,

        "mean_intercept_delay_s":
            mean_delay,

        "total_intercept_delay_s":
            total_delay,

        "unique_bands_found":
            unique_bands,
    }


In [11]:
# ============================================================
# LOAD ORACLE SCHEDULE
# ============================================================

TRAIN_ORACLE_DIR = Path("./Oracle")
VAL_ORACLE_DIR = Path("./val_oracle")


def load_oracle_schedule(
    environment_name,
    oracle_dir
):

    oracle_dir = Path(
        oracle_dir
    )

    schedule_path = (
        oracle_dir
        / environment_name.replace(
            ".npy",
            "_oracle_schedule.npy"
        )
    )

    if not schedule_path.exists():

        raise FileNotFoundError(
            f"Oracle schedule not found:\n"
            f"{schedule_path.resolve()}\n\n"
            f"Environment: {environment_name}\n"
            f"Oracle directory: {oracle_dir.resolve()}"
        )

    schedule = np.load(
        schedule_path
    ).astype(np.int64)

    return schedule

In [12]:

# ============================================================
# 37. FINAL EVALUATION HELPERS
# ============================================================

def extract_policy_metrics(
    result
):

    return {
        "total_events":
            result[
                "total_event_count"
            ],

        "intercepted_events":
            result[
                "intercepted_event_count"
            ],

        "missed_events":
            len(
                result[
                    "missed_events"
                ]
            ),

        "interception_rate":
            result[
                "event_interception_rate"
            ],

        "mean_delay_s":
            result[
                "mean_intercept_delay_s"
            ],

        "total_delay_s":
            result[
                "total_intercept_delay_s"
            ],

        "unique_bands":
            result[
                "unique_bands_found"
            ],
    }


def evaluate_policy(
    policy,
    environments
):

    results = []

    for path in environments:

        env = np.load(
            path
        ).astype(np.float32)

        result = run_policy_on_environment(
            env=env,
            policy=policy,
            belief_provider=belief_provider,
            reward_function=None,
            training=False,
        )

        metrics = extract_policy_metrics(
            result
        )

        metrics["environment"] = (
            path.name
        )

        results.append({
            "path": path,
            "result": result,
            "metrics": metrics,
        })

    return results


In [13]:

# ============================================================
# 38. ROBUST ORACLE EVENT INTERCEPTIONS
# ============================================================

def oracle_event_interceptions(
    env,
    oracle_schedule
):
    """
    Reconstruct event-level Oracle interceptions.

    The Oracle schedule is evaluated only over the timesteps for
    which both the environment and schedule exist.

    A longer Oracle file therefore cannot cause an IndexError.
    """

    env = np.asarray(
        env,
        dtype=np.float32
    )

    schedule = np.asarray(
        oracle_schedule,
        dtype=np.int64
    ).reshape(-1)

    events = extract_transmission_events(
        env
    )

    event_by_id = {
        event["id"]: event
        for event in events
    }

    active_events = build_event_lookup(
        events,
        len(env)
    )

    evaluation_steps = min(
        len(env),
        len(schedule)
    )

    intercepted_ids = set()
    interception_times = {}

    for t in range(
        evaluation_steps
    ):

        band = int(
            schedule[t]
        )

        if band < 0:
            continue

        if band >= N_FREQ_BINS:
            continue

        for event_id in active_events[t]:

            if event_id in intercepted_ids:
                continue

            event = event_by_id[
                event_id
            ]

            if event["band"] == band:

                intercepted_ids.add(
                    event_id
                )

                interception_times[
                    event_id
                ] = t

    intercepted = {}

    for event_id, t in (
        interception_times.items()
    ):

        event = event_by_id[
            event_id
        ]

        delay_steps = (
            t - event["start"]
        )

        intercepted[event_id] = {
            **event,

            "interception_time":
                t,

            "delay_steps":
                delay_steps,

            "delay_s":
                delay_steps * TIME_BIN_S,
        }

    return intercepted


def compare_to_oracle(
    env,
    environment_name,
    policy_result,
    oracle_dir
):
    """
    Compare the frozen policy with the saved Oracle.

    IMPORTANT:
        The policy result is generated before this function is
        called. The Oracle therefore cannot influence action
        selection.

    Metrics:
        policy_isr:
            policy intercepted events / all environment events.

        oracle_isr:
            Oracle intercepted events / all environment events.

        capture_vs_oracle:
            policy intercepted / Oracle intercepted.

        common-event delay:
            compare delay only for events intercepted by both.
    """

    env = np.asarray(
        env,
        dtype=np.float32
    )

    oracle_schedule = (
        load_oracle_schedule(
            environment_name,
            oracle_dir
         )
    )

    schedule_length = len(
        oracle_schedule
    )

    if schedule_length != len(env):

        print(
            f"WARNING: {environment_name}: "
            f"environment has {len(env)} steps, "
            f"Oracle has {schedule_length} steps. "
            f"Evaluating the first "
            f"{min(len(env), schedule_length)} steps."
        )

    oracle_events = (
        oracle_event_interceptions(
            env,
            oracle_schedule
        )
    )

    policy_events = {
        event["id"]: event
        for event in policy_result[
            "intercepted_events"
        ]
    }

    total_events = len(
        extract_transmission_events(env)
    )

    policy_intercepted = len(
        policy_events
    )

    oracle_intercepted = len(
        oracle_events
    )

    policy_isr = (
        policy_intercepted / total_events
        if total_events > 0
        else np.nan
    )

    oracle_isr = (
        oracle_intercepted / total_events
        if total_events > 0
        else np.nan
    )

    capture_vs_oracle = (
        policy_intercepted
        / oracle_intercepted
        if oracle_intercepted > 0
        else np.nan
    )

    common_ids = (
        set(policy_events)
        &
        set(oracle_events)
    )

    if common_ids:

        policy_delays = np.asarray([
            policy_events[i]["delay_s"]
            for i in common_ids
        ])

        oracle_delays = np.asarray([
            oracle_events[i]["delay_s"]
            for i in common_ids
        ])

        policy_common_delay = float(
            np.mean(policy_delays)
        )

        oracle_common_delay = float(
            np.mean(oracle_delays)
        )

        excess_delay = float(
            policy_common_delay
            - oracle_common_delay
        )

        delay_overhead_pct = (
            excess_delay
            / oracle_common_delay
            * 100.0
            if oracle_common_delay > 0
            else np.nan
        )

    else:

        policy_common_delay = np.nan
        oracle_common_delay = np.nan
        excess_delay = np.nan
        delay_overhead_pct = np.nan

    return {
        "environment":
            environment_name,

        "total_events":
            total_events,

        "policy_intercepted":
            policy_intercepted,

        "oracle_intercepted":
            oracle_intercepted,

        "policy_isr":
            policy_isr,

        "oracle_isr":
            oracle_isr,

        "capture_vs_oracle":
            capture_vs_oracle,

        "common_events":
            len(common_ids),

        "policy_common_mean_delay_s":
            policy_common_delay,

        "oracle_common_mean_delay_s":
            oracle_common_delay,

        "mean_excess_delay_s":
            excess_delay,

        "delay_overhead_pct":
            delay_overhead_pct,

        "oracle_schedule_length":
            schedule_length,

        "environment_length":
            len(env),
    }


In [14]:
# ============================================================
# 45. COUNTERFACTUAL ORACLE-TO-BANDIT MEGA SWEEP

# ONE-CELL MULTI-EXPERIMENT LEARNING STUDY

# Main architecture preserved:

#     RF environment
#          ↓
#     frozen recursive belief
#          ↓
#     temporal scheduler state
#          ↓
#     GRU contextual neural bandit
#          ↓
#     Thompson / UCB / mean-Q

# What changes:

#     DATA GENERATION
#     ORACLE SUPERVISION
#     COUNTERFACTUAL LEARNING
#     HORIZON
#     HARD-NEGATIVE BALANCING
#     POLICY EXPLORATION

# The FULL RF environment is used only OFFLINE to construct
# teacher targets.

# During validation the policy receives ONLY:

#     current belief
#     scheduler memory
#     selected-band observation

# ============================================================


import copy
import gc
import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


print("=" * 110)
print("E45 INITIALIZATION: COUNTERFACTUAL H1 TRAINING")
print("=" * 110)


# ============================================================
# 0. SAFETY CHECKS
# ============================================================

_REQUIRED_45 = [

    "N_FREQ_BINS",
    "STATE_DIM",
    "CANDIDATE_DIM",
    "SEQUENCE_LENGTH",

    "NeuralBanditHead",
    "TemporalStateBuilder",
    "RecursiveBeliefProvider",

    "recursive_model",
    "P_baseline",
    "DEVICE",

    "TRAIN_RFENV_DIR",
    "VAL_RFENV_DIR",
    "TRAIN_ORACLE_DIR",
    "VAL_ORACLE_DIR",

    "list_rf_environments",
    "load_oracle_schedule",

    "run_policy_on_environment",
    "extract_policy_metrics",
    "compare_to_oracle",

    "extract_transmission_events",
    "build_event_lookup",

    "TIME_BIN_S",
    "MAX_SCAN_GAP_STEPS",
]


_missing_45 = [
    x
    for x in _REQUIRED_45
    if x not in globals()
]


if _missing_45:

    raise RuntimeError(
        "Missing required notebook objects:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in _missing_45
        )
    )


train_files_45 = (
    list_rf_environments(
        TRAIN_RFENV_DIR
    )
)

val_files_45 = (
    list_rf_environments(
        VAL_RFENV_DIR
    )
)


print(
    f"Training environments   : "
    f"{len(train_files_45)}"
)

print(
    f"Validation environments : "
    f"{len(val_files_45)}"
)

print(
    f"Bands                   : "
    f"{N_FREQ_BINS}"
)

print(
    f"State dimension         : "
    f"{STATE_DIM}"
)

print(
    f"Candidate dimension     : "
    f"{CANDIDATE_DIM}"
)

print(
    f"Sequence length         : "
    f"{SEQUENCE_LENGTH}"
)

print(
    f"Device                  : "
    f"{DEVICE}"
)


# ============================================================
# 1. COUNTERFACTUAL DATASET CONFIGURATION
# ============================================================

E45_HORIZONS = (
    1,
    4,
    8,
    16
)

E45_BEHAVIORS = (

    "oracle",

    "cyclic",

    "random",

    "belief",

    "uncertainty",
)

E45_DELAY_SCALE = 4.0

E45_MAX_STATES_PER_ENV_BEHAVIOR = 700

E45_SEED = 45001


# ============================================================
# 2. ORACLE LOADER
# ============================================================

def _safe_oracle_45(
    path
):

    try:

        schedule = np.asarray(
            load_oracle_schedule(
                path.name,
                TRAIN_ORACLE_DIR
            )
        ).reshape(-1)

        schedule = np.where(

            (
                (schedule >= 0)
                &
                (schedule < N_FREQ_BINS)
            ),

            schedule,

            -1
        )

        return schedule.astype(
            np.int64
        )

    except Exception:

        env = np.load(
            path
        )

        return np.full(
            len(env),
            -1,
            dtype=np.int64
        )


# ============================================================
# 3. COUNTERFACTUAL TEACHER

# For every information state and every possible action,
# estimate how useful that action would be using OFFLINE
# ground truth.

# This is NOT available during deployment.

# H=1:
#     immediate interception opportunity

# H=4/8/16:
#     delay-aware future interception opportunity
# ============================================================

def _teacher_values_45(
    env,
    t,
    active_events,
    event_by_id,
    unseen_events
):

    n = len(env)

    targets = {

        h:
            np.zeros(
                N_FREQ_BINS,
                dtype=np.float32
            )

        for h in E45_HORIZONS
    }


    for horizon in E45_HORIZONS:

        end = min(
            n,
            t + horizon
        )

        for future_t in range(
            t,
            end
        ):

            delay = (
                future_t
                - t
            )

            decay = math.exp(
                -delay
                /
                E45_DELAY_SCALE
            )

            for event_id in (
                active_events[
                    future_t
                ]
            ):

                if event_id not in (
                    unseen_events
                ):

                    continue

                event = (
                    event_by_id[
                        event_id
                    ]
                )

                band = int(
                    event["band"]
                )

                targets[
                    horizon
                ][band] = max(

                    targets[
                        horizon
                    ][band],

                    decay
                )


    return targets


# ============================================================
# 4. OFFLINE BEHAVIOR POLICIES
# ============================================================

def _behavior_action_45(

    mode,

    t,

    belief,

    uncertainty,

    oracle,

    rng
):

    # --------------------------------------------------------
    # ORACLE
    # --------------------------------------------------------

    if mode == "oracle":

        if t < len(
            oracle
        ):

            action = int(
                oracle[t]
            )

            if (
                0
                <= action
                < N_FREQ_BINS
            ):

                return action

        return int(
            t % N_FREQ_BINS
        )


    # --------------------------------------------------------
    # CYCLIC
    # --------------------------------------------------------

    if mode == "cyclic":

        return int(
            t % N_FREQ_BINS
        )


    # --------------------------------------------------------
    # RANDOM
    # --------------------------------------------------------

    if mode == "random":

        return int(
            rng.integers(
                0,
                N_FREQ_BINS
            )
        )


    # --------------------------------------------------------
    # BELIEF GREEDY
    # --------------------------------------------------------

    if mode == "belief":

        if rng.random() < 0.15:

            return int(
                rng.integers(
                    0,
                    N_FREQ_BINS
                )
            )

        return int(
            np.argmax(
                belief
            )
        )


    # --------------------------------------------------------
    # UNCERTAINTY
    # --------------------------------------------------------

    if mode == "uncertainty":

        score = (
            uncertainty
            *
            (
                0.25
                +
                belief
            )
        )

        if rng.random() < 0.15:

            return int(
                rng.integers(
                    0,
                    N_FREQ_BINS
                )
            )

        return int(
            np.argmax(
                score
            )
        )


    raise ValueError(
        f"Unknown behavior mode: {mode}"
    )


# ============================================================
# 5. COLLECT ONE OFFLINE BEHAVIOR TRAJECTORY
# ============================================================

def _collect_behavior_45(
    path,
    mode,
    seed
):

    env = np.asarray(
        np.load(path),
        dtype=np.float32
    )


    events = (
        extract_transmission_events(
            env
        )
    )


    active_events = (
        build_event_lookup(
            events,
            len(env)
        )
    )


    event_by_id = {

        event["id"]:
            event

        for event in events
    }


    oracle = (
        _safe_oracle_45(
            path
        )
    )


    provider = (
        RecursiveBeliefProvider(

            recursive_model,

            P_baseline,

            DEVICE
        )
    )


    builder = (
        TemporalStateBuilder(

            N_FREQ_BINS,

            SEQUENCE_LENGTH
        )
    )


    provider.reset()

    builder.reset()


    rng = np.random.default_rng(
        seed
    )


    belief = provider(

        t=0,

        action=None,

        observation=None
    )


    uncertainty = (
        4.0
        * belief
        * (
            1.0
            - belief
        )
    )


    # Events not yet intercepted by
    # this behavior trajectory.

    unseen_events = set(

        event["id"]

        for event in events
    )


    rows = []

    n_steps = len(
        env
    )


    # --------------------------------------------------------
    # Downsample ONLY if needed.
    # --------------------------------------------------------

    if (
        n_steps
        >
        E45_MAX_STATES_PER_ENV_BEHAVIOR
    ):

        keep_indices = (

            np.linspace(

                0,

                n_steps - 1,

                E45_MAX_STATES_PER_ENV_BEHAVIOR

            )
            .astype(int)
        )

        keep_set = set(
            int(x)
            for x
            in keep_indices
        )

    else:

        keep_set = set(
            range(n_steps)
        )


    # ========================================================
    # ONLINE-STYLE TRAJECTORY
    # ========================================================

    for t in range(
        n_steps
    ):

        # ----------------------------------------------------
        # Information state BEFORE action.
        # ----------------------------------------------------

        sequence, candidates = (
            builder.observe_state(
                belief
            )
        )


        # ----------------------------------------------------
        # OFFLINE COUNTERFACTUAL TEACHER.
        #
        # This uses complete RF truth, but ONLY for
        # constructing the offline training target.
        # ----------------------------------------------------

        teacher = (
            _teacher_values_45(

                env,

                t,

                active_events,

                event_by_id,

                unseen_events
            )
        )


        # ----------------------------------------------------
        # Record state.
        # ----------------------------------------------------

        if t in keep_set:

            rows.append({

                "state":
                    sequence.astype(
                        np.float16
                    ),

                "candidate":
                    candidates.astype(
                        np.float16
                    ),

                "oracle":
                    (
                        int(
                            oracle[t]
                        )
                        if t < len(
                            oracle
                        )
                        else -1
                    ),

                "teacher_h1":
                    teacher[1],

                "teacher_h4":
                    teacher[4],

                "teacher_h8":
                    teacher[8],

                "teacher_h16":
                    teacher[16],
            })


        # ----------------------------------------------------
        # Choose behavior action.
        # ----------------------------------------------------

        action = (
            _behavior_action_45(

                mode,

                t,

                belief,

                uncertainty,

                oracle,

                rng
            )
        )


        # ----------------------------------------------------
        # ONLY selected-band observation.
        # ----------------------------------------------------

        observation = float(
            env[
                t,
                action
            ]
        )


        # ----------------------------------------------------
        # Update frozen belief.
        # ----------------------------------------------------

        belief = provider(

            t=t + 1,

            action=action,

            observation=observation
        )


        uncertainty = (

            4.0
            * belief
            * (
                1.0
                - belief
            )
        )


        # ----------------------------------------------------
        # Update scheduler memory.
        # ----------------------------------------------------

        builder.update(
            action,
            observation
        )


        # ----------------------------------------------------
        # Offline event bookkeeping.
        # ----------------------------------------------------

        for event_id in (
            active_events[t]
        ):

            event = (
                event_by_id[
                    event_id
                ]
            )

            if (
                event["band"]
                ==
                action
            ):

                unseen_events.discard(
                    event_id
                )


    return rows


# ============================================================
# 6. BUILD THE DATASET
# ============================================================

print()
print("=" * 100)
print("BUILDING COUNTERFACTUAL INFORMATION-STATE DATASET")
print("=" * 100)


all_rows_45 = []


for env_index, path in enumerate(

    train_files_45,

    start=1

):

    before = len(
        all_rows_45
    )


    for mode_index, mode in enumerate(
        E45_BEHAVIORS
    ):

        rows = (
            _collect_behavior_45(

                path,

                mode,

                E45_SEED
                +
                env_index * 100
                +
                mode_index
            )
        )

        all_rows_45.extend(
            rows
        )


    added = (
        len(all_rows_45)
        -
        before
    )


    print(
        f"[{env_index}/"
        f"{len(train_files_45)}] "
        f"{path.name}: "
        f"+{added} states"
    )


if not all_rows_45:

    raise RuntimeError(
        "No training states were created."
    )


# ============================================================
# 7. CONVERT DATASET TO ARRAYS
# ============================================================

states_45 = np.stack(

    [
        row["state"]
        for row
        in all_rows_45
    ]
)


candidates_45 = np.stack(

    [
        row["candidate"]
        for row
        in all_rows_45
    ]
)


oracle_actions_45 = np.asarray(

    [
        row["oracle"]
        for row
        in all_rows_45
    ],

    dtype=np.int64
)


teacher_h1_45 = np.stack(

    [
        row["teacher_h1"]
        for row
        in all_rows_45
    ]

).astype(
    np.float32
)


teacher_h4_45 = np.stack(

    [
        row["teacher_h4"]
        for row
        in all_rows_45
    ]

).astype(
    np.float32
)


teacher_h8_45 = np.stack(

    [
        row["teacher_h8"]
        for row
        in all_rows_45
    ]

).astype(
    np.float32
)


teacher_h16_45 = np.stack(

    [
        row["teacher_h16"]
        for row
        in all_rows_45
    ]

).astype(
    np.float32
)


valid_oracle_45 = (

    (oracle_actions_45 >= 0)

    &

    (
        oracle_actions_45
        < N_FREQ_BINS
    )
)


print()
print(
    f"Information states : "
    f"{len(states_45):,}"
)

print(
    f"Counterfactual actions : "
    f"{len(states_45) * N_FREQ_BINS:,}"
)

print(
    f"Valid Oracle labels : "
    f"{int(valid_oracle_45.sum()):,} "
    f"("
    f"{valid_oracle_45.mean():.1%}"
    ")"
)

print(
    f"Invalid Oracle labels : "
    f"{int((~valid_oracle_45).sum()):,}"
)


# ============================================================
# 8. EXPERIMENT CONFIGURATIONS

# SAME MODEL ARCHITECTURE.

# ONLY THE LEARNING PROCEDURE CHANGES.
# ============================================================

EXPERIMENTS_45 = {

    # This is the exact E45 learning regime that produced
    # the established ~29.09% E45 reference.
    "C_counterfactual_H1": {

        "target":
            "h1",

        "bc":
            0.0,

        "rank":
            0.0,

        "reg":
            1.0,

        "soft":
            0.0,

        "hard_negative":
            False,

        "lr":
            7e-4,

        "epochs":
            4,
    },
}


# ============================================================
# 9. TRAINING ENGINE
# ============================================================

E45_HEADS = 3

E45_BATCH_SIZE = 128

E45_TEMPERATURE = 0.20

E45_RANK_MARGIN = 0.20

E45_WEIGHT_DECAY = 1e-4

E45_GRAD_CLIP = 5.0


def _get_teacher_45(
    name
):

    return {

        "h1":
            teacher_h1_45,

        "h4":
            teacher_h4_45,

        "h8":
            teacher_h8_45,

        "h16":
            teacher_h16_45,

    }[name]


def _train_experiment_45(
    name,
    config
):

    print()
    print("-" * 105)
    print(
        f"TRAINING: {name}"
    )
    print("-" * 105)


    # Deterministic per experiment.

    experiment_seed = (

        E45_SEED

        +

        sum(
            (
                index + 1
            )
            *
            ord(char)

            for index, char
            in enumerate(name)
        )

        % 100000
    )


    torch.manual_seed(
        experiment_seed
    )

    np.random.seed(
        experiment_seed
    )


    heads = [

        NeuralBanditHead()
        .to(DEVICE)

        for _ in range(
            E45_HEADS
        )

    ]


    optimizers = [

        optim.AdamW(

            head.parameters(),

            lr=config["lr"],

            weight_decay=(
                E45_WEIGHT_DECAY
            )

        )

        for head
        in heads

    ]


    target = (
        _get_teacher_45(
            config["target"]
        )
    )


    # Convert counterfactual target to
    # a probability distribution.

    teacher_prob = (

        target

        /

        (
            target.sum(
                axis=1,
                keepdims=True
            )
            + 1e-6
        )

    )


    n = len(
        states_45
    )


    history = []


    for epoch in range(
        config["epochs"]
    ):

        order = np.random.permutation(
            n
        )

        epoch_losses = []


        for start in range(

            0,

            n,

            E45_BATCH_SIZE

        ):

            indices = order[
                start:
                start + E45_BATCH_SIZE
            ]


            state_tensor = torch.as_tensor(

                states_45[
                    indices
                ],

                dtype=torch.float32,

                device=DEVICE
            )


            candidate_tensor = torch.as_tensor(

                candidates_45[
                    indices
                ],

                dtype=torch.float32,

                device=DEVICE
            )


            teacher_tensor = torch.as_tensor(

                target[
                    indices
                ],

                dtype=torch.float32,

                device=DEVICE
            )


            teacher_prob_tensor = torch.as_tensor(

                teacher_prob[
                    indices
                ],

                dtype=torch.float32,

                device=DEVICE
            )


            oracle_tensor = torch.as_tensor(

                oracle_actions_45[
                    indices
                ],

                dtype=torch.long,

                device=DEVICE
            )


            valid = (

                (oracle_tensor >= 0)

                &

                (
                    oracle_tensor
                    < N_FREQ_BINS
                )
            )


            for head, optimizer in zip(

                heads,

                optimizers

            ):

                q_values = head(

                    state_tensor,

                    candidate_tensor

                )


                total_loss = torch.zeros(

                    (),

                    device=DEVICE
                )


                # ====================================================
                # A. ORACLE BEHAVIOR CLONING
                # ====================================================

                if (

                    config["bc"] > 0

                    and

                    torch.any(valid)

                ):

                    bc_loss = (
                        F.cross_entropy(

                            q_values[
                                valid
                            ],

                            oracle_tensor[
                                valid
                            ]

                        )
                    )

                    total_loss = (

                        total_loss

                        +

                        config["bc"]
                        *
                        bc_loss

                    )


                # ====================================================
                # B. ORACLE RANKING
                # ====================================================

                if (

                    config["rank"] > 0

                    and

                    torch.any(valid)

                ):

                    valid_q = (
                        q_values[
                            valid
                        ]
                    )


                    valid_oracle = (
                        oracle_tensor[
                            valid
                        ]
                    )


                    oracle_q = (

                        valid_q

                        .gather(

                            1,

                            valid_oracle.unsqueeze(
                                1
                            )

                        )

                        .squeeze(1)

                    )


                    negative_q = (
                        valid_q.clone()
                    )


                    negative_q.scatter_(

                        1,

                        valid_oracle.unsqueeze(
                            1
                        ),

                        -torch.inf

                    )


                    hardest_negative = (

                        negative_q

                        .max(
                            dim=1
                        )
                        .values

                    )


                    ranking_loss = (

                        F.relu(

                            E45_RANK_MARGIN

                            -

                            oracle_q

                            +

                            hardest_negative

                        )

                        .mean()

                    )


                    total_loss = (

                        total_loss

                        +

                        config["rank"]
                        *
                        ranking_loss

                    )


                # ====================================================
                # C. COUNTERFACTUAL VALUE REGRESSION
                # ====================================================

                if config["reg"] > 0:

                    element_loss = (

                        F.smooth_l1_loss(

                            q_values,

                            teacher_tensor,

                            reduction="none"

                        )

                    )


                    if config["hard_negative"]:

                        weights = (

                            1.0

                            +

                            4.0
                            *
                            teacher_tensor.detach()

                        )

                        regression_loss = (

                            element_loss
                            *
                            weights

                        ).mean()

                    else:

                        regression_loss = (
                            element_loss.mean()
                        )


                    total_loss = (

                        total_loss

                        +

                        config["reg"]
                        *
                        regression_loss

                    )


                # ====================================================
                # D. SOFT COUNTERFACTUAL DISTILLATION
                # ====================================================

                if config["soft"] > 0:

                    log_probability = (

                        F.log_softmax(

                            q_values
                            /
                            E45_TEMPERATURE,

                            dim=1

                        )

                    )


                    soft_loss = (

                        F.kl_div(

                            log_probability,

                            teacher_prob_tensor,

                            reduction="batchmean"

                        )

                    )


                    total_loss = (

                        total_loss

                        +

                        config["soft"]
                        *
                        soft_loss

                    )


                optimizer.zero_grad(
                    set_to_none=True
                )


                total_loss.backward()


                torch.nn.utils.clip_grad_norm_(

                    head.parameters(),

                    E45_GRAD_CLIP

                )


                optimizer.step()


                epoch_losses.append(

                    float(
                        total_loss
                        .detach()
                        .cpu()
                    )

                )


        mean_epoch_loss = float(
            np.mean(
                epoch_losses
            )
        )


        history.append(
            mean_epoch_loss
        )


        print(

            f"epoch "
            f"{epoch + 1}/"
            f"{config['epochs']} "
            f"loss="
            f"{mean_epoch_loss:.6f}"

        )


    for head in heads:

        head.eval()


    return {

        "heads":
            heads,

        "history":
            history,

        "config":
            config,

    }


# ============================================================

# ============================================================
# FINAL E45 INITIALIZATION
# Train ONLY the established E45 starting regime used by E52.
# No E45 experiment sweep and no E45 validation sweep.
# ============================================================

FINAL_E45_CONFIG_NAME = "C_counterfactual_H1"

if FINAL_E45_CONFIG_NAME not in EXPERIMENTS_45:
    raise RuntimeError(
        f"Required E45 configuration {FINAL_E45_CONFIG_NAME!r} is missing."
    )

print()
print("=" * 100)
print("TRAINING FINAL E45 INITIALIZATION (C_counterfactual_H1)")
print("=" * 100)
print(f"Configuration: {FINAL_E45_CONFIG_NAME}")

best_learning_model_45 = _train_experiment_45(
    FINAL_E45_CONFIG_NAME,
    EXPERIMENTS_45[FINAL_E45_CONFIG_NAME],
)

e45_init_state_52 = [
    copy.deepcopy(head.state_dict())
    for head in best_learning_model_45["heads"]
]

e45_init_config_52 = copy.deepcopy(
    best_learning_model_45["config"]
)

print()
print("✓ E45 C_counterfactual_H1 initialization trained in memory.")
print("✓ No E45 checkpoint written to disk.")


E45 INITIALIZATION: COUNTERFACTUAL H1 TRAINING
Training environments   : 5
Validation environments : 13
Bands                   : 36
State dimension         : 254
Candidate dimension     : 7
Sequence length         : 12
Device                  : cuda

BUILDING COUNTERFACTUAL INFORMATION-STATE DATASET
[1/5] config_0.npy: +2990 states
[2/5] config_1.npy: +2985 states
[3/5] config_10.npy: +3355 states
[4/5] config_100.npy: +3000 states
[5/5] config_1000.npy: +2885 states

Information states : 15,215
Counterfactual actions : 547,740
Valid Oracle labels : 6,915 (45.4%)
Invalid Oracle labels : 8,300

TRAINING FINAL E45 INITIALIZATION (C_counterfactual_H1)
Configuration: C_counterfactual_H1

---------------------------------------------------------------------------------------------------------
TRAINING: C_counterfactual_H1
---------------------------------------------------------------------------------------------------------
epoch 1/4 loss=0.026116
epoch 2/4 loss=0.016697
epoch 3/4 loss=0

In [15]:
# ==========================================================================================================
# EXPERIMENT 52: COUNTERFACTUAL OFFLINE RL + EVENT-AWARE DOUBLE-Q

# IMPORTANT:
#   This is a COMPLETE replacement cell.

# Major fixes:
#   1. Counterfactual observation is now env[t, action], NOT the whole RF row.
#   2. Correct [B,A] -> [B*A] belief-update shapes.
#   3. True Double-Q next-action selection:
#        online network chooses argmax_a Q(s',a)
#        target network evaluates that action.
#   4. Event-aware counterfactual reward.
#   5. Multi-step information-state learning.
#   6. Experiment-45 anchor prevents catastrophic forgetting.
#   7. Validation is untouched during training.

# ==========================================================================================================

import gc
import math
import time
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


E52_START = time.time()

print("=" * 110)
print("FINAL E52 TRAINING: COUNTERFACTUAL OFFLINE RL + EVENT-AWARE DOUBLE-Q")
print("=" * 110)


# ==========================================================================================================
# 0. REQUIRED NOTEBOOK OBJECTS
# ==========================================================================================================

_REQUIRED_E52 = [
    "N_FREQ_BINS",
    "STATE_DIM",
    "CANDIDATE_DIM",
    "SEQUENCE_LENGTH",
    "GRU_HIDDEN_DIM",
    "ACTION_EMBED_DIM",
    "Q_HIDDEN_DIM",
    "NeuralBanditHead",
    "TemporalStateBuilder",
    "RecursiveBeliefProvider",
    "recursive_model",
    "P_baseline",
    "DEVICE",
    "TRAIN_RFENV_DIR",
    "VAL_RFENV_DIR",
    "TRAIN_FINAL_DIR",
    "VAL_ORACLE_DIR",
    "list_rf_environments",
    "run_policy_on_environment",
    "extract_policy_metrics",
    "compare_to_oracle",
    "extract_transmission_events",
    "build_event_lookup",
    "TIME_BIN_S",
    "MAX_SCAN_GAP_STEPS",
]

_missing_E52 = [
    x for x in _REQUIRED_E52
    if x not in globals()
]

if _missing_E52:

    raise RuntimeError(
        "Experiment 52 cannot start.\n\n"
        "Missing notebook objects:\n"
        + "\n".join(
            f"  - {x}"
            for x in _missing_E52
        )
        + "\n\n"
        "Run the notebook from the beginning through the base scheduler cells."
    )


# ==========================================================================================================
# 1. DATA / CHECKPOINTS
# ==========================================================================================================

train_files_52 = list_rf_environments(
    TRAIN_RFENV_DIR
)

val_files_52 = list_rf_environments(
    VAL_RFENV_DIR
)

print(
    f"Training environments   : {len(train_files_52)}"
)

print(
    f"Validation environments : {len(val_files_52)}"
)

print(
    f"Bands                   : {N_FREQ_BINS}"
)

print(
    f"State dimension         : {STATE_DIM}"
)

print(
    f"Candidate dimension     : {CANDIDATE_DIM}"
)

print(
    f"Sequence length         : {SEQUENCE_LENGTH}"
)

print(
    f"Device                  : {DEVICE}"
)


# Experiment 45 was trained immediately above and is retained in memory.
# This avoids writing the intermediate E45 model to the user's hard drive.
if "e45_init_state_52" not in globals():
    raise RuntimeError(
        "Experiment 45 in-memory initialization is missing. "
        "Run this notebook from the beginning so E45 completes before E52."
    )

base_states_52 = e45_init_state_52

E52_SOURCE_CHECKPOINT = None

E52_OUTPUT_CHECKPOINT = (
    TRAIN_FINAL_DIR
    / "experiment52_counterfactual_offline_rl_best.pt"
)

E52_HEADS = min(
    3,
    len(base_states_52)
)

if E52_HEADS < 1:
    raise RuntimeError(
        "Experiment-45 initialization contains zero heads."
    )

print("✓ E45 C_counterfactual_H1 initialization loaded from memory")
print(f"✓ Ensemble heads: {E52_HEADS}")


# ==========================================================================================================
# 2. EXPERIMENT CONFIGURATION
# ==========================================================================================================

E52_SEED = 52052

E52_BATCH = 96

E52_EPOCHS = 2

E52_FQI_ROUNDS = 3

E52_LR = 1e-4

E52_WEIGHT_DECAY = 1e-4

E52_GAMMA = 0.985

E52_GRAD_CLIP = 3.0


# More states than the broken version,
# but still controlled enough to avoid enormous runtime.
E52_MAX_STATES_PER_ENV = 1600

E52_STRIDE = 2


# Long enough for meaningful future value,
# but not so long that the target becomes extremely noisy.
E52_HORIZON = 8


# Event reward shaping.
E52_EVENT_DECAY = 5.0

E52_EVENT_BONUS = 1.0

E52_FUTURE_BONUS = 0.30


# Penalize repeatedly scanning recently scanned bands.
E52_REDUNDANCY_PENALTY = 0.015


# Auxiliary action ranking.
E52_RANK_WEIGHT = 0.20


# Conservative regularization.
E52_CQL_WEIGHT = 0.03

E52_CQL_TEMP = 1.0


# Prevent the new RL model from destroying Exp45.
E52_ANCHOR_WEIGHT = 0.10


# Bootstrap mask.
E52_BOOTSTRAP_P = 0.80


# Deployment policy.
E52_INITIAL_SWEEP = True

E52_STALE_BONUS = 0.12


np.random.seed(
    E52_SEED
)

torch.manual_seed(
    E52_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        E52_SEED
    )


print()
print("Configuration:")
print(
    f"  Heads                 : {E52_HEADS}"
)
print(
    f"  Batch                 : {E52_BATCH}"
)
print(
    f"  Epochs / FQI          : {E52_EPOCHS}"
)
print(
    f"  FQI rounds            : {E52_FQI_ROUNDS}"
)
print(
    f"  Gamma                 : {E52_GAMMA}"
)
print(
    f"  Counterfactual horizon: {E52_HORIZON}"
)
print(
    f"  Event bonus           : {E52_EVENT_BONUS}"
)
print(
    f"  Future bonus          : {E52_FUTURE_BONUS}"
)
print(
    f"  Ranking weight        : {E52_RANK_WEIGHT}"
)
print(
    f"  CQL weight            : {E52_CQL_WEIGHT}"
)
print(
    f"  Exp45 anchor          : {E52_ANCHOR_WEIGHT}"
)


# ==========================================================================================================
# 3. LOAD TRAINING ENVIRONMENTS
# ==========================================================================================================

E52_BEHAVIORS = (
    "cyclic",
    "random",
    "belief",
)


def _e52_behavior_action(
    mode,
    t,
    belief,
    rng
):

    if mode == "cyclic":

        return int(
            t % N_FREQ_BINS
        )


    if mode == "random":

        return int(
            rng.integers(
                0,
                N_FREQ_BINS
            )
        )


    if mode == "belief":

        if rng.random() < 0.20:

            return int(
                rng.integers(
                    0,
                    N_FREQ_BINS
                )
            )

        return int(
            np.argmax(
                np.asarray(
                    belief
                )
            )
        )


    raise ValueError(
        f"Unknown behavior: {mode}"
    )


# ==========================================================================================================
# 4. COLLECT REAL INFORMATION STATES
# ==========================================================================================================

def _e52_collect(
    path,
    mode,
    seed
):

    env = np.asarray(
        np.load(path),
        dtype=np.float32
    )


    events = extract_transmission_events(
        env
    )

    active_events = build_event_lookup(
        events,
        len(env)
    )

    provider = RecursiveBeliefProvider(
        recursive_model,
        P_baseline,
        DEVICE
    )

    builder = TemporalStateBuilder(
        N_FREQ_BINS,
        SEQUENCE_LENGTH
    )


    provider.reset()

    builder.reset()


    rng = np.random.default_rng(
        seed
    )


    belief = provider(
        t=0,
        action=None,
        observation=None
    )


    unseen = set(
        e["id"]
        for e in events
    )


    n = len(env)


    if n <= E52_MAX_STATES_PER_ENV:

        keep = set(
            range(
                0,
                n,
                E52_STRIDE
            )
        )

    else:

        idx = np.linspace(
            0,
            n - 1,
            E52_MAX_STATES_PER_ENV
        ).astype(
            int
        )

        keep = set(
            int(x)
            for x in idx
        )


    rows = []


    for t in range(n):

        sequence, candidates = (
            builder.observe_state(
                belief
            )
        )


        if t in keep:

            rows.append(
                {
                    "sequence":
                        sequence.astype(
                            np.float16
                        ),

                    "candidate":
                        candidates.astype(
                            np.float16
                        ),

                    "belief":
                        np.asarray(
                            belief,
                            dtype=np.float16
                        ),

                    "t":
                        int(t),

                    "env_path":
                        str(path),

                    "unseen":
                        tuple(
                            sorted(
                                unseen
                            )
                        ),
                }
            )


        action = _e52_behavior_action(
            mode,
            t,
            belief,
            rng
        )


        observation = float(
            env[
                t,
                action
            ]
        )


        belief = provider(
            t=t + 1,
            action=action,
            observation=observation
        )


        builder.update(
            action,
            observation
        )


        for event_id in active_events[t]:

            if (
                event_by_id_band :=
                events[event_id]["band"]
            ) == action:

                unseen.discard(
                    event_id
                )


    return rows


# ==========================================================================================================
# 5. BUILD DATASET
# ==========================================================================================================

print()
print("=" * 100)
print("BUILDING OFFLINE INFORMATION-STATE DATASET")
print("=" * 100)


all_rows_52 = []


for ei, path in enumerate(
    train_files_52,
    1
):

    before = len(
        all_rows_52
    )


    for bi, mode in enumerate(
        E52_BEHAVIORS
    ):

        all_rows_52.extend(
            _e52_collect(
                path,
                mode,
                E52_SEED
                + ei * 100
                + bi
            )
        )


    print(
        f"[{ei:03d}/{len(train_files_52):03d}] "
        f"{path.name}: "
        f"+{len(all_rows_52) - before:,} states"
    )


if not all_rows_52:

    raise RuntimeError(
        "Experiment 52 produced zero training states."
    )


print()
print(
    f"Total offline states : "
    f"{len(all_rows_52):,}"
)


# ==========================================================================================================
# 6. PACK DATASET
# ==========================================================================================================

states_52 = np.stack(
    [
        r["sequence"]
        for r in all_rows_52
    ]
).astype(
    np.float16
)


candidates_52 = np.stack(
    [
        r["candidate"]
        for r in all_rows_52
    ]
).astype(
    np.float16
)


beliefs_52 = np.stack(
    [
        r["belief"]
        for r in all_rows_52
    ]
).astype(
    np.float32
)


times_52 = np.asarray(
    [
        r["t"]
        for r in all_rows_52
    ],
    dtype=np.int32
)


paths_52 = np.asarray(
    [
        r["env_path"]
        for r in all_rows_52
    ]
)


unseen_52 = [
    r["unseen"]
    for r in all_rows_52
]


print(
    f"State tensor         : "
    f"{states_52.shape}"
)

print(
    f"Candidate tensor     : "
    f"{candidates_52.shape}"
)

print(
    f"Counterfactual acts  : "
    f"{len(states_52) * N_FREQ_BINS:,}"
)


# ==========================================================================================================
# 7. CACHE RF ENVIRONMENTS
# ==========================================================================================================

env_cache_52 = {}


for p in np.unique(
    paths_52
):

    env_cache_52[p] = np.asarray(
        np.load(p),
        dtype=np.float32
    )


events_cache_52 = {}

active_cache_52 = {}


for p, env in env_cache_52.items():

    ev = extract_transmission_events(
        env
    )

    events_cache_52[p] = ev

    active_cache_52[p] = (
        build_event_lookup(
            ev,
            len(env)
        )
    )


# ==========================================================================================================
# 8. COUNTERFACTUAL EVENT REWARD
# ==========================================================================================================

def _e52_reward_matrix(
    indices
):

    B = len(
        indices
    )

    A = N_FREQ_BINS


    R = np.zeros(
        (
            B,
            A
        ),
        dtype=np.float32
    )


    for bi, idx in enumerate(
        indices
    ):

        p = paths_52[idx]

        env = env_cache_52[p]

        events = events_cache_52[p]

        active_lookup = (
            active_cache_52[p]
        )

        t = int(
            times_52[idx]
        )

        unseen = set(
            unseen_52[idx]
        )


        current_by_band = [
            []
            for _ in range(A)
        ]


        for eid in active_lookup[t]:

            if eid in unseen:

                e = events[eid]

                current_by_band[
                    e["band"]
                ].append(
                    e
                )


        future_best = np.zeros(
            A,
            dtype=np.float32
        )


        future_end = min(
            len(env),
            t + E52_HORIZON
        )


        for tt in range(
            t + 1,
            future_end
        ):

            delay = float(
                tt - t
            )

            decay = math.exp(
                -delay
                / E52_EVENT_DECAY
            )


            for eid in active_lookup[tt]:

                if eid not in unseen:

                    continue


                band = events[eid]["band"]


                future_best[band] = max(
                    future_best[band],
                    decay
                )


        for a in range(A):

            obs = float(
                env[t, a]
            )


            value = 0.0


            # ------------------------------------------------
            # IMMEDIATE EVENT CAPTURE
            # ------------------------------------------------

            if (
                obs > 0
                and current_by_band[a]
            ):

                best_now = 0.0


                for e in current_by_band[a]:

                    age = max(
                        0,
                        t - int(
                            e["start"]
                        )
                    )


                    best_now = max(
                        best_now,
                        math.exp(
                            -age
                            / E52_EVENT_DECAY
                        )
                    )


                value += (
                    E52_EVENT_BONUS
                    * (
                        1.0
                        + best_now
                    )
                )


            # ------------------------------------------------
            # FUTURE EVENT VALUE
            # ------------------------------------------------

            value += (
                E52_FUTURE_BONUS
                * future_best[a]
            )


            # ------------------------------------------------
            # REDUNDANCY COST
            # ------------------------------------------------

            age_norm = float(
                candidates_52[
                    idx,
                    a,
                    1
                ]
            )


            value -= (
                E52_REDUNDANCY_PENALTY
                * (
                    1.0
                    - age_norm
                )
            )


            R[
                bi,
                a
            ] = value


    return R


# ==========================================================================================================
# 9. CORRECT COUNTERFACTUAL NEXT-STATE GENERATION

# THIS IS THE CRITICAL FIX.

# For each state b and hypothetical action a:

    observation[b,a] = env_b[t, a]

# NOT the complete RF row.

# Thus:

#     belief input : [B*A, 36]
    action       : [B*A]
    observation  : [B*A]

# ==========================================================================================================

@torch.no_grad()
def _e52_counterfactual_next(
    indices,
    actions_tensor
):

    B = len(
        indices
    )

    A = N_FREQ_BINS

    L = SEQUENCE_LENGTH

    D = STATE_DIM


    actions_np = (
        actions_tensor
        .detach()
        .cpu()
        .numpy()
        .astype(
            np.int64
        )
    )


    if actions_np.shape != (
        B,
        A
    ):

        raise RuntimeError(
            "Counterfactual action shape error: "
            f"expected {(B,A)}, "
            f"got {actions_np.shape}"
        )


    # ------------------------------------------------------------------
    # CURRENT BELIEFS
    # ------------------------------------------------------------------

    current_belief = torch.as_tensor(
        beliefs_52[
            np.asarray(indices)
        ],
        dtype=torch.float32,
        device=DEVICE
    )


    # [B,36] -> [B,36,36] -> [B*36,36]
    bel = (
        current_belief
        .unsqueeze(1)
        .expand(
            B,
            A,
            A
        )
        .reshape(
            B * A,
            A
        )
    )


    # ------------------------------------------------------------------
    # CRITICAL FIX:
    #
    # For every hypothetical action, retrieve ONLY the observation
    # from that action's band.
    # ------------------------------------------------------------------

    obs_np = np.zeros(
        (
            B,
            A
        ),
        dtype=np.float32
    )


    for b, idx in enumerate(
        indices
    ):

        env = env_cache_52[
            paths_52[idx]
        ]

        t = int(
            times_52[idx]
        )


        # env[t, :] is 36 observations.
        #
        # actions_np[b, :] contains the 36 hypothetical actions.
        #
        # Therefore:
        #
        # obs_np[b,a] = env[t, actions_np[b,a]]
        #
        obs_np[b, :] = (
            env[
                t,
                actions_np[b, :]
            ]
        )


    # [B,36] -> [B*36]
    obs = torch.as_tensor(
        obs_np.reshape(-1),
        dtype=torch.float32,
        device=DEVICE
    )


    # [B,36] -> [B*36]
    act = torch.as_tensor(
        actions_np.reshape(-1),
        dtype=torch.long,
        device=DEVICE
    )


    # ------------------------------------------------------------------
    # HARD SHAPE CHECK
    # ------------------------------------------------------------------

    assert bel.shape == (
        B * A,
        A
    )

    assert act.shape == (
        B * A,
    )

    assert obs.shape == (
        B * A,
    )


    # ------------------------------------------------------------------
    # RECURSIVE BELIEF UPDATE
    # ------------------------------------------------------------------

    next_belief = torch.sigmoid(
        recursive_model(
            bel,
            act,
            obs
        )
    )


    next_belief = next_belief.reshape(
        B,
        A,
        A
    )


    # ------------------------------------------------------------------
    # CURRENT LAST STATE
    # ------------------------------------------------------------------

    last = states_52[
        np.asarray(indices),
        -1
    ].astype(
        np.float32
    )


    next_last = np.repeat(
        last[:, None, :],
        A,
        axis=1
    )


    # ------------------------------------------------------------------
    # STATE BLOCKS
    # ------------------------------------------------------------------

    s0 = 0

    age0 = A

    hit0 = 2 * A

    tsh0 = 3 * A

    obs0 = 4 * A

    scan0 = 5 * A

    prev0 = 6 * A

    time0 = 7 * A


    max_gap = float(
        MAX_SCAN_GAP_STEPS
    )


    # ------------------------------------------------------------------
    # AGE / TIME SINCE HIT
    # ------------------------------------------------------------------

    next_last[
        :,
        :,
        age0:age0+A
    ] = np.minimum(
        last[
            :,
            None,
            age0:age0+A
        ]
        + 1.0 / max_gap,
        1.0
    )


    next_last[
        :,
        :,
        tsh0:tsh0+A
    ] = np.minimum(
        last[
            :,
            None,
            tsh0:tsh0+A
        ]
        + 1.0 / max_gap,
        1.0
    )


    rows = np.arange(
        B
    )[:, None]

    cols = np.arange(
        A
    )[None, :]


    # ------------------------------------------------------------------
    # SELECTED ACTION BECOMES FRESHLY SCANNED
    # ------------------------------------------------------------------

    next_last[
        rows,
        cols,
        age0 + actions_np
    ] = 0.0


    # ------------------------------------------------------------------
    # HIT / LAST OBSERVATION
    # ------------------------------------------------------------------

    hit = (
        obs_np > 0
    ).astype(
        np.float32
    )


    next_last[
        rows,
        cols,
        tsh0 + actions_np
    ] = np.where(
        hit > 0,
        0.0,
        next_last[
            rows,
            cols,
            tsh0 + actions_np
        ]
    )


    next_last[
        rows,
        cols,
        hit0 + actions_np
    ] = (
        0.9
        * next_last[
            rows,
            cols,
            hit0 + actions_np
        ]
        + 0.1 * hit
    )


    next_last[
        rows,
        cols,
        obs0 + actions_np
    ] = hit


    # ------------------------------------------------------------------
    # SCAN COUNTS
    # ------------------------------------------------------------------

    scan_norm = last[
        :,
        None,
        scan0:scan0+A
    ]


    raw_scan = np.expm1(
        np.clip(
            scan_norm,
            0.0,
            1.0
        )
        * np.log1p(50.0)
    )


    raw_scan = (
        raw_scan
        + 1.0
    )


    next_scan = np.clip(
        np.log1p(
            raw_scan
        )
        / np.log1p(50.0),
        0.0,
        1.0
    )


    next_last[
        :,
        :,
        scan0:scan0+A
    ] = next_scan


    # ------------------------------------------------------------------
    # PREVIOUS ACTION
    # ------------------------------------------------------------------

    next_last[
        :,
        :,
        prev0:prev0+A
    ] = 0.0


    next_last[
        rows,
        cols,
        prev0 + actions_np
    ] = 1.0


    # ------------------------------------------------------------------
    # TIME ENCODING
    # ------------------------------------------------------------------

    tvals = (
        np.asarray(
            times_52[
                np.asarray(indices)
            ],
            dtype=np.float32
        )[:, None]
        + 1.0
    )


    phase = (
        2.0
        * np.pi
        * tvals
        / 200.0
    )


    phase_stack = np.stack(
        [
            np.sin(phase),
            np.cos(phase)
        ],
        axis=-1
    )


    next_last[
        :,
        :,
        time0:time0+2
    ] = phase_stack


    # ------------------------------------------------------------------
    # NEW BELIEF
    # ------------------------------------------------------------------

    next_last[
        :,
        :,
        s0:s0+A
    ] = (
        next_belief
        .detach()
        .cpu()
        .numpy()
    )


    # ------------------------------------------------------------------
    # NEXT CANDIDATE FEATURES
    # ------------------------------------------------------------------

    next_candidates = np.zeros(
        (
            B,
            A,
            A,
            CANDIDATE_DIM
        ),
        dtype=np.float32
    )


    next_candidates[
        :,
        :,
        :,
        0
    ] = next_last[
        :,
        :,
        s0:s0+A
    ]


    next_candidates[
        :,
        :,
        :,
        1
    ] = next_last[
        :,
        :,
        age0:age0+A
    ]


    next_candidates[
        :,
        :,
        :,
        2
    ] = next_last[
        :,
        :,
        hit0:hit0+A
    ]


    next_candidates[
        :,
        :,
        :,
        3
    ] = next_last[
        :,
        :,
        tsh0:tsh0+A
    ]


    next_candidates[
        :,
        :,
        :,
        4
    ] = next_last[
        :,
        :,
        obs0:obs0+A
    ]


    next_candidates[
        :,
        :,
        :,
        5
    ] = next_last[
        :,
        :,
        scan0:scan0+A
    ]


    next_candidates[
        :,
        :,
        :,
        6
    ] = (
        np.arange(
            A,
            dtype=np.float32
        )[None, None, :]
        / max(
            A - 1,
            1
        )
    )


    # ------------------------------------------------------------------
    # NEXT SEQUENCE
    # ------------------------------------------------------------------

    current_seq = states_52[
        np.asarray(indices)
    ].astype(
        np.float32
    )


    next_seq = np.repeat(
        current_seq[:, None, :, :],
        A,
        axis=1
    )


    next_seq[
        :,
        :,
        :-1,
        :
    ] = current_seq[
        :,
        None,
        1:,
        :
    ]


    next_seq[
        :,
        :,
        -1,
        :
    ] = next_last


    # ------------------------------------------------------------------
    # TERMINAL FLAGS
    # ------------------------------------------------------------------

    done = np.zeros(
        (
            B,
            A
        ),
        dtype=np.float32
    )


    for b, idx in enumerate(
        indices
    ):

        p = paths_52[idx]

        env_len = len(
            env_cache_52[p]
        )

        done[b, :] = float(
            int(
                times_52[idx]
            )
            >= env_len - 1
        )


    # ------------------------------------------------------------------
    # FINAL SHAPE CHECKS
    # ------------------------------------------------------------------

    assert next_seq.shape == (
        B,
        A,
        L,
        D
    )


    assert next_candidates.shape == (
        B,
        A,
        A,
        CANDIDATE_DIM
    )


    assert done.shape == (
        B,
        A
    )


    return (
        next_seq.astype(
            np.float16
        ),
        next_candidates.astype(
            np.float16
        ),
        done
    )


# ==========================================================================================================
# 10. TEST COUNTERFACTUAL GENERATOR BEFORE TRAINING
# ==========================================================================================================

print()
print("=" * 100)
print("COUNTERFACTUAL SHAPE SANITY CHECK")
print("=" * 100)


_test_B = min(
    4,
    len(states_52)
)


_test_indices = np.arange(
    _test_B,
    dtype=np.int64
)


_test_actions = torch.arange(
    N_FREQ_BINS,
    device=DEVICE
).unsqueeze(0).expand(
    _test_B,
    -1
)


_test_ns, _test_nc, _test_done = (
    _e52_counterfactual_next(
        _test_indices,
        _test_actions
    )
)


print(
    f"Next sequence shape     : "
    f"{_test_ns.shape}"
)

print(
    f"Next candidate shape    : "
    f"{_test_nc.shape}"
)

print(
    f"Done shape              : "
    f"{_test_done.shape}"
)


assert _test_ns.shape == (
    _test_B,
    N_FREQ_BINS,
    SEQUENCE_LENGTH,
    STATE_DIM
)


assert _test_nc.shape == (
    _test_B,
    N_FREQ_BINS,
    N_FREQ_BINS,
    CANDIDATE_DIM
)


assert _test_done.shape == (
    _test_B,
    N_FREQ_BINS
)


print(
    "✓ Counterfactual observation shape is correct."
)

print(
    "✓ Recursive belief update shape is correct."
)

print(
    "✓ Next-state tensors are correct."
)

print(
    "✓ Training will NOT start unless these checks pass."
)


# ==========================================================================================================
# 11. CREATE HEAD
# ==========================================================================================================

def _new_head(
    seed_offset
):

    torch.manual_seed(
        E52_SEED
        + seed_offset
    )


    head = NeuralBanditHead().to(
        DEVICE
    )


    return head


# ==========================================================================================================
# 12. FIT ONE DOUBLE-Q HEAD
# ==========================================================================================================

def _train_e52_head(
    head,
    target_head,
    head_id
):

    optimizer = optim.AdamW(
        head.parameters(),
        lr=E52_LR,
        weight_decay=E52_WEIGHT_DECAY
    )


    head.train()

    target_head.eval()


    n = len(
        states_52
    )


    history = []


    # ======================================================================
    # FQI ROUNDS
    # ======================================================================

    for fqi_round in range(
        E52_FQI_ROUNDS
    ):

        print()
        print(
            "-" * 100
        )

        print(
            f"HEAD {head_id + 1}/{E52_HEADS} "
            f"| FQI ROUND {fqi_round + 1}/{E52_FQI_ROUNDS}"
        )

        print(
            "-" * 100
        )


        # ------------------------------------------------------------------
        # Build Bellman targets.
        #
        # TRUE DOUBLE-Q:
        #
        #   a* = argmax_a Q_online(s',a)
        #
        #   target = r + gamma Q_target(s',a*)
        #
        # ------------------------------------------------------------------

        target_values = np.zeros(
            (
                n,
                N_FREQ_BINS
            ),
            dtype=np.float32
        )


        for start in range(
            0,
            n,
            E52_BATCH
        ):

            idx = np.arange(
                start,
                min(
                    start + E52_BATCH,
                    n
                )
            )


            B = len(
                idx
            )


            rewards = _e52_reward_matrix(
                idx
            )


            actions = torch.arange(
                N_FREQ_BINS,
                device=DEVICE
            ).unsqueeze(
                0
            ).expand(
                B,
                -1
            )


            (
                next_seq_np,
                next_cand_np,
                done_np
            ) = _e52_counterfactual_next(
                idx,
                actions
            )


            ns = torch.as_tensor(
                next_seq_np,
                dtype=torch.float32,
                device=DEVICE
            ).reshape(
                B * N_FREQ_BINS,
                SEQUENCE_LENGTH,
                STATE_DIM
            )


            nc = torch.as_tensor(
                next_cand_np,
                dtype=torch.float32,
                device=DEVICE
            ).reshape(
                B * N_FREQ_BINS,
                N_FREQ_BINS,
                CANDIDATE_DIM
            )


            with torch.no_grad():

                # ----------------------------------------------------------
                # ONLINE network selects the action.
                # ----------------------------------------------------------

                q_online = head(
                    ns,
                    nc
                ).reshape(
                    B,
                    N_FREQ_BINS,
                    N_FREQ_BINS
                )


                best_actions = (
                    q_online
                    .argmax(
                        dim=2
                    )
                )


                # ----------------------------------------------------------
                # TARGET network evaluates the selected action.
                # ----------------------------------------------------------

                q_target = target_head(
                    ns,
                    nc
                ).reshape(
                    B,
                    N_FREQ_BINS,
                    N_FREQ_BINS
                )


                best_next = (
                    q_target
                    .gather(
                        2,
                        best_actions.unsqueeze(-1)
                    )
                    .squeeze(-1)
                )


                reward_tensor = torch.as_tensor(
                    rewards,
                    dtype=torch.float32,
                    device=DEVICE
                )


                done_tensor = torch.as_tensor(
                    done_np,
                    dtype=torch.float32,
                    device=DEVICE
                )


                y = (
                    reward_tensor
                    + E52_GAMMA
                    * (
                        1.0
                        - done_tensor
                    )
                    * best_next
                )


                target_values[
                    idx
                ] = (
                    y
                    .detach()
                    .cpu()
                    .numpy()
                )


        target_values = np.clip(
            target_values,
            -20.0,
            20.0
        )


        # ------------------------------------------------------------------
        # OPTIMIZE
        # ------------------------------------------------------------------

        for epoch in range(
            E52_EPOCHS
        ):

            order = np.random.default_rng(
                E52_SEED
                + head_id * 10000
                + fqi_round * 100
                + epoch
            ).permutation(
                n
            )


            losses = []

            td_losses = []

            rank_losses = []

            cql_losses = []

            anchor_losses = []


            for start in range(
                0,
                n,
                E52_BATCH
            ):

                idx = order[
                    start:
                    start + E52_BATCH
                ]


                seq = torch.as_tensor(
                    states_52[idx],
                    dtype=torch.float32,
                    device=DEVICE
                )


                cand = torch.as_tensor(
                    candidates_52[idx],
                    dtype=torch.float32,
                    device=DEVICE
                )


                y = torch.as_tensor(
                    target_values[idx],
                    dtype=torch.float32,
                    device=DEVICE
                )


                q = head(
                    seq,
                    cand
                )


                # ----------------------------------------------------------
                # Bellman regression
                # ----------------------------------------------------------

                td_loss = F.smooth_l1_loss(
                    q,
                    y
                )


                # ----------------------------------------------------------
                # Exp45 preservation anchor
                # ----------------------------------------------------------

                with torch.no_grad():

                    base_head = _new_head(
                        1000
                        + head_id
                    )

                    base_head.load_state_dict(
                        base_states_52[
                            head_id
                        ]
                    )

                    base_head.eval()

                    base_q = base_head(
                        seq,
                        cand
                    )


                anchor_loss = F.smooth_l1_loss(
                    q,
                    base_q
                )


                del base_head


                # ----------------------------------------------------------
                # Best-action ranking
                # ----------------------------------------------------------

                top_action = (
                    y.argmax(
                        dim=1
                    )
                )


                q_top = q.gather(
                    1,
                    top_action.unsqueeze(
                        1
                    )
                ).squeeze(
                    1
                )


                q_without_top = q.clone()


                q_without_top.scatter_(
                    1,
                    top_action.unsqueeze(
                        1
                    ),
                    -torch.inf
                )


                q_negative = (
                    q_without_top
                    .max(
                        dim=1
                    )
                    .values
                )


                rank_loss = F.relu(
                    0.10
                    - q_top
                    + q_negative
                ).mean()


                # ----------------------------------------------------------
                # Conservative Q regularization
                # ----------------------------------------------------------

                cql_loss = (
                    torch.logsumexp(
                        q
                        / E52_CQL_TEMP,
                        dim=1
                    )
                    * E52_CQL_TEMP
                    - q.mean(
                        dim=1
                    )
                ).mean()


                # ----------------------------------------------------------
                # Combined objective
                # ----------------------------------------------------------

                loss = (
                    td_loss
                    + E52_RANK_WEIGHT
                    * rank_loss
                    + E52_CQL_WEIGHT
                    * cql_loss
                    + E52_ANCHOR_WEIGHT
                    * anchor_loss
                )


                optimizer.zero_grad(
                    set_to_none=True
                )


                loss.backward()


                torch.nn.utils.clip_grad_norm_(
                    head.parameters(),
                    E52_GRAD_CLIP
                )


                optimizer.step()


                losses.append(
                    float(
                        loss.detach()
                        .cpu()
                    )
                )

                td_losses.append(
                    float(
                        td_loss.detach()
                        .cpu()
                    )
                )

                rank_losses.append(
                    float(
                        rank_loss.detach()
                        .cpu()
                    )
                )

                cql_losses.append(
                    float(
                        cql_loss.detach()
                        .cpu()
                    )
                )

                anchor_losses.append(
                    float(
                        anchor_loss.detach()
                        .cpu()
                    )
                )


            mean_loss = float(
                np.mean(
                    losses
                )
            )


            history.append(
                mean_loss
            )


            print(
                f"  epoch {epoch + 1}/{E52_EPOCHS}"
                f" | total={mean_loss:.6f}"
                f" | TD={np.mean(td_losses):.6f}"
                f" | rank={np.mean(rank_losses):.6f}"
                f" | CQL={np.mean(cql_losses):.6f}"
                f" | anchor={np.mean(anchor_losses):.6f}"
            )


        # ------------------------------------------------------------------
        # SOFT TARGET UPDATE
        # ------------------------------------------------------------------

        with torch.no_grad():

            for tp, p in zip(
                target_head.parameters(),
                head.parameters()
            ):

                tp.data.mul_(
                    0.90
                ).add_(
                    0.10 * p.data
                )


    head.eval()

    return history


# ==========================================================================================================
# 13. TRAIN ENSEMBLE
# ==========================================================================================================

print()
print("=" * 110)
print("TRAINING EVENT-AWARE DOUBLE-Q ENSEMBLE")
print("=" * 110)


heads_52 = []

target_heads_52 = []

histories_52 = []


for h in range(
    E52_HEADS
):

    print()
    print(
        "=" * 90
    )

    print(
        f"TRAINING HEAD {h + 1}/{E52_HEADS}"
    )

    print(
        "=" * 90
    )


    head = _new_head(
        h
    )


    head.load_state_dict(
        base_states_52[h]
    )


    target = _new_head(
        100 + h
    )


    target.load_state_dict(
        base_states_52[h]
    )


    history = _train_e52_head(
        head,
        target,
        h
    )


    heads_52.append(
        head
    )

    target_heads_52.append(
        target
    )

    histories_52.append(
        history
    )


print()
print(
    "✓ Event-aware Double-Q ensemble trained."
)


# ==========================================================================================================

# ============================================================
# SAVE FINAL E52 MODEL
# No E52 policy sweep or validation sweep is run here.
# finalPrototype_E52.ipynb performs deployment/evaluation.
# ============================================================

E52_OUTPUT_CHECKPOINT = (
    TRAIN_FINAL_DIR
    / "experiment52_counterfactual_offline_rl_best.pt"
)

torch.save(
    {
        "experiment": 52,
        "source_checkpoint": "E45 in-memory initialization",
        "model_state_dict": [
            h.state_dict()
            for h in heads_52
        ],
        "target_state_dict": [
            h.state_dict()
            for h in target_heads_52
        ],
        "results": [],
        "config": {
            "gamma": E52_GAMMA,
            "horizon": E52_HORIZON,
            "event_bonus": E52_EVENT_BONUS,
            "future_bonus": E52_FUTURE_BONUS,
            "rank_weight": E52_RANK_WEIGHT,
            "cql_weight": E52_CQL_WEIGHT,
            "anchor_weight": E52_ANCHOR_WEIGHT,
            "stale_bonus": E52_STALE_BONUS,
            "heads": E52_HEADS,
            "fqi_rounds": E52_FQI_ROUNDS,
            "epochs": E52_EPOCHS,
        },
        "architecture": {
            "state_dim": STATE_DIM,
            "candidate_dim": CANDIDATE_DIM,
            "sequence_length": SEQUENCE_LENGTH,
            "n_bands": N_FREQ_BINS,
        },
    },
    E52_OUTPUT_CHECKPOINT,
)

print()
print("=" * 100)
print("FINAL E52 MODEL SAVED")
print("=" * 100)
print(f"Checkpoint: {E52_OUTPUT_CHECKPOINT.resolve()}")
print("✓ Only the final E52 checkpoint is written to disk.")
print("✓ E45 exists only in memory.")
print("✓ No validation/policy experiment sweep was run.")


FINAL E52 TRAINING: COUNTERFACTUAL OFFLINE RL + EVENT-AWARE DOUBLE-Q
Training environments   : 5
Validation environments : 13
Bands                   : 36
State dimension         : 254
Candidate dimension     : 7
Sequence length         : 12
Device                  : cuda
✓ E45 C_counterfactual_H1 initialization loaded from memory
✓ Ensemble heads: 3

Configuration:
  Heads                 : 3
  Batch                 : 96
  Epochs / FQI          : 2
  FQI rounds            : 3
  Gamma                 : 0.985
  Counterfactual horizon: 8
  Event bonus           : 1.0
  Future bonus          : 0.3
  Ranking weight        : 0.2
  CQL weight            : 0.03
  Exp45 anchor          : 0.1

BUILDING OFFLINE INFORMATION-STATE DATASET
[001/005] config_0.npy: +897 states
[002/005] config_1.npy: +897 states
[003/005] config_10.npy: +1,008 states
[004/005] config_100.npy: +900 states
[005/005] config_1000.npy: +867 states

Total offline states : 4,569
State tensor         : (4569, 12, 254)
Candid